# 0. Imports

## 0.1 Packages

In [132]:
import asyncio
import nest_asyncio
from tenacity import retry, wait_exponential, stop_after_attempt
import aiohttp
import pandas as pd

## 0.2 Data

In [133]:
ObsList = pd.read_csv(r"../Data Raw/ObsList.csv", sep=";")

# 1. Taxonomy Uniformization

## 1.1. Code

### 1.1.1. GBIF: Names Check

#### 1.1.1.1. Uniformization

In [134]:
nest_asyncio.apply()

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))
async def GBIF_Species_1(session, species):

    url = f"https://api.gbif.org/v1/species/match?name={species}"  
    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()
            
            if data.get('rank') == 'SPECIES':
                if data.get('status') != 'ACCEPTED':
                    accepted_name = data.get('species') or data.get('canonicalName')
                else:
                    accepted_name = data.get('canonicalName')
                
                return accepted_name if accepted_name else None
            else:
                return species
    except Exception as e:
        print(f"Request failed for species '{species}': {e}")
        return None

async def GBIF_Sessions_1(species_list):
    async with aiohttp.ClientSession() as session:
        tasks = [GBIF_Species_1(session, species) for species in species_list]
        return await asyncio.gather(*tasks)

def GBIF_Extractor_1(species_list):
    return asyncio.get_event_loop().run_until_complete(GBIF_Sessions_1(species_list))

#### 1.1.1.2. Filter

In [135]:
nest_asyncio.apply()

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))
async def GBIF_Species_2(session, species):

    url = f"https://api.gbif.org/v1/species/match?name={species}"  
    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()
            
            if data.get('rank') == 'SPECIES':
                if data.get('status') != 'ACCEPTED':
                    accepted_name = data.get('species') or data.get('canonicalName')
                else:
                    accepted_name = data.get('canonicalName')
                
                return accepted_name if accepted_name else None
            else:
                return None
    except Exception as e:
        print(f"Request failed for species '{species}': {e}")
        return None

async def GBIF_Sessions_2(species_list):
    async with aiohttp.ClientSession() as session:
        tasks = [GBIF_Species_2(session, species) for species in species_list]
        return await asyncio.gather(*tasks)

def GBIF_Extractor_2(species_list):
    return asyncio.get_event_loop().run_until_complete(GBIF_Sessions_2(species_list))

### 1.1.2. Global Names Verifier: Cross-check

#### 1.1.2.1. Uniformization

In [136]:
nest_asyncio.apply()

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))
async def VNF_Species_1(session, species):
    
    url = f"https://verifier.globalnames.org/api/v1/verifications/{species}?data_sources=1%7C12&all_matches=false&capitalize=false&species_group=false&fuzzy_uninomial=false&stats=true&main_taxon_threshold=0.5"
    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()
            
            if data.get("names", [{}])[0].get("bestResult", {}).get("taxonomicStatus") == 'Accepted':
                accepted_name = data.get("names", [{}])[0].get("bestResult", {}).get('matchedCanonicalSimple')
            else:
                accepted_name = data.get("names", [{}])[0].get("bestResult", {}).get('currentCanonicalSimple')
            return accepted_name if accepted_name != "" else species
            
            
    except Exception as e:
        print(f"Request failed for species '{species}': {e}")
        return None

async def VNF_Sessions_1(species_list):
    async with aiohttp.ClientSession() as session:
        tasks = [VNF_Species_1(session, species) for species in species_list]
        return await asyncio.gather(*tasks)

def VNF_Extractor_1(species_list):
    return asyncio.get_event_loop().run_until_complete(VNF_Sessions_1(species_list))

#### 1.1.2.2. Filter

In [137]:
nest_asyncio.apply()

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))
async def VNF_Species_2(session, species):
    
    url = f"https://verifier.globalnames.org/api/v1/verifications/{species}?data_sources=1%7C12&all_matches=false&capitalize=false&species_group=false&fuzzy_uninomial=false&stats=true&main_taxon_threshold=0.5"
    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()
            
            if data.get("names", [{}])[0].get("bestResult", {}).get("taxonomicStatus") == 'Accepted':
                accepted_name = data.get("names", [{}])[0].get("bestResult", {}).get('matchedCanonicalSimple')
            else:
                accepted_name = data.get("names", [{}])[0].get("bestResult", {}).get('currentCanonicalSimple')
            return accepted_name
            
            
    except Exception as e:
        print(f"Request failed for species '{species}': {e}")
        return None

async def VNF_Sessions_2(species_list):
    async with aiohttp.ClientSession() as session:
        tasks = [VNF_Species_2(session, species) for species in species_list]
        return await asyncio.gather(*tasks)

def VNF_Extractor_2(species_list):
    return asyncio.get_event_loop().run_until_complete(VNF_Sessions_2(species_list))

### 1.1.3. GBIF: Family Extract

In [138]:
nest_asyncio.apply()

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))
async def GBIF_Family(session, species):
    url = f"https://api.gbif.org/v1/species/match?name={species}"
    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()
            
            if data.get('family'):
                return data.get('family')
            else:
                return None
    
    except Exception as e:
        print(f"Request failed for species '{species}': {e}")
        return None

async def GBIF_Family_Sessions(species_list):
    async with aiohttp.ClientSession() as session:
        tasks = [GBIF_Family(session, species) for species in species_list]
        return await asyncio.gather(*tasks)

def GBIF_Family_Extract(species_list):
    return asyncio.get_event_loop().run_until_complete(GBIF_Family_Sessions(species_list))

### 1.1.4. GBIF: Lepidoptera Cross-check

In [139]:
nest_asyncio.apply()

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))
async def GBIF_Lepidoptera(session, family):
    
    url = f"https://api.gbif.org/v1/species/match?name={family}"
    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()
            
            if data.get('order') == 'Lepidoptera':
                return 1
            elif data.get('order') is None:
                return None
            else:
                return 0
            
    except Exception as e:
        print(f"Error fetching data for {family}: {e}")
        return False

async def GBIF_Lepidoptera_Sessions(family_list):
    async with aiohttp.ClientSession() as session:
        tasks = [GBIF_Lepidoptera(session, family) for family in family_list]
        return await asyncio.gather(*tasks)

def GBIF_Lepidoptera_Extract(family_list):
    return asyncio.get_event_loop().run_until_complete(GBIF_Lepidoptera_Sessions(family_list))

## 1.2. Data

In [140]:
ObsList['Species'] = ObsList['Species'].apply(lambda x: ' '.join(x.split()[:2]))

In [141]:
ObsList['VNF'] = VNF_Extractor_1(ObsList['Species'])
ObsList['VNF'] = ObsList['VNF'].apply(lambda x: ' '.join(x.split()[:2]) if isinstance(x, str) else x)

In [142]:
ObsList['AcceptedSpecies'] = GBIF_Extractor_2(ObsList['VNF'])
ObsList['AcceptedSpecies'] = ObsList['AcceptedSpecies'].apply(lambda x: ' '.join(x.split()[:2]) if isinstance(x, str) else x)

In [143]:
ObsList[ObsList['AcceptedSpecies'].isna()]['Species'].unique()

array(['Stylopalpis lunigerella', 'Heliocheilus cystiphora',
       'Rheumaptera affirmata', 'Spodoptera sunia', 'Heliocontia margana',
       'Trissodoris guamensis', 'Prospalta dolorosa',
       'Phalaenophana fadusalis', 'Opodiphthera eucalypti',
       'Eodiatraea rufescens', 'Euphaedra temeraria', 'Orgyia basalis'],
      dtype=object)

In [144]:
ObsList[ObsList['AcceptedSpecies'].isna()].shape[0]


16

In [145]:
ObservationsRaw = ObsList.copy()
ObservationsRaw.dropna(subset=['AcceptedSpecies'], inplace=True)
ObservationsRaw[ObservationsRaw['AcceptedSpecies'].isna()]

,Species,NAME_0,Realm,Cryptogenic,Dispersal,Eradicated,Intentional_Release,Introduced,Established,Observation Year,Reference,Reference Year,VNF,AcceptedSpecies


In [146]:
ObservationsRaw.reset_index(drop=True, inplace=True)
ObservationsRaw.to_csv(r'../Transformed Data/ObservationsRaw.csv', sep =';')

# 2. Taxonomy Table Extract

In [147]:
Taxonomy = pd.DataFrame()
Taxonomy['Species'] = [*ObservationsRaw['Species'], *ObservationsRaw['AcceptedSpecies']]
Taxonomy.drop_duplicates(subset=['Species'], inplace=True)
Taxonomy.reset_index(drop=True, inplace=True)

## 2.1 Accepted Species

In [148]:
Taxonomy['AcceptedSpecies'] = GBIF_Extractor_2(VNF_Extractor_1(Taxonomy['Species']))

## 2.2. Genus

In [149]:
Taxonomy['Genus'] = Taxonomy['AcceptedSpecies'].astype(str).str.split().str[0]


## 2.3. Family

In [150]:
Taxonomy['Family'] = GBIF_Family_Extract(Taxonomy['AcceptedSpecies'])
Taxonomy.reset_index(drop=True, inplace=True)

In [156]:
Taxonomy[Taxonomy['Family'].isna()]

,Species,AcceptedSpecies,Genus,Family


## 2.4. Lepidoptera Cross-Check

In [152]:
Taxonomy['Lepidoptera'] = GBIF_Lepidoptera_Extract(Taxonomy['Family'])
Taxonomy.reset_index(drop=True, inplace=True)

In [153]:
Taxonomy[(Taxonomy['Lepidoptera'].isna()) | (Taxonomy['Lepidoptera'] == 0)]['Family'].unique()

array(['Saturniidae', None], dtype=object)

## 2.5 Manual Updates

In [154]:
Taxonomy.loc[Taxonomy['AcceptedSpecies'] == 'Homoeographa lanceolella', 'Family'] = 'Pyralidae'
Taxonomy.loc[Taxonomy['AcceptedSpecies'] == 'Calephelis virginiensis', 'Family'] = 'Riodinidae'

As: Riodinidae, Pyralidae and Saturniidae are all Lepidopterans - drop Lepidoptera crosscheck column

In [155]:
Taxonomy.drop(columns=['Lepidoptera'], inplace=True)

## 2.6 Export

In [157]:
Taxonomy.to_csv(r'../Data Raw/TaxonomyRaw.csv', index=False)

# 3. Natives Taxonomy

In [199]:
Natives = pd.read_csv(r'../Data Raw/NativeRaw.csv', sep=';')

In [200]:
Natives.drop('AcceptedSpecies', axis=1, inplace=True)

In [204]:
Natives['AcceptedSpecies'] = check_species(Natives['Species'])
Natives['AcceptedSpecies'] = Natives['AcceptedSpecies'].fillna(Natives['Species'])


In [206]:
Natives.to_csv(r'../Data Raw/NativeRaw.csv', sep=';')